In [22]:
import pandas as pd
import numpy as np
from datetime import datetime
from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly

In [3]:
df = pd.read_csv('../../data/03-final/daily_truck_data.csv')
df.head()

,order_date,truck_count,cycle_time,volume
0,2023-03-16,9,71.962963,115.5
1,2023-03-17,8,70.692308,80.5
2,2023-03-18,3,65.750000,13.5
3,2023-03-19,0,0.000000,0.0
4,2023-03-20,0,0.000000,0.0


In [4]:
# only useful columns
columns_to_keep = ['order_date', 'volume']
filtered_df = df[columns_to_keep]
filtered_df.head()

,order_date,volume
0,2023-03-16,115.5
1,2023-03-17,80.5
2,2023-03-18,13.5
3,2023-03-19,0.0
4,2023-03-20,0.0


# Prophet model

In [5]:
# rename to ds and y
prophet_df = filtered_df.rename(columns={'order_date': 'ds', 'volume': 'y'})
prophet_df.head()

,ds,y
0,2023-03-16,115.5
1,2023-03-17,80.5
2,2023-03-18,13.5
3,2023-03-19,0.0
4,2023-03-20,0.0


## define holidays

In [6]:
# ── Define holidays ────────────────────────────────────────────────────────────
# Adjust / extend this list to match your actual calendar.
# Format: name (str) + ds (date) — one row per non-operating day.

holiday_dates = [
    # --- Mexico public holidays (examples) ---
    '2023-01-01', '2023-02-06', '2023-03-20', '2023-05-01', '2023-09-16',
    '2023-11-20', '2023-12-25',
    '2024-01-01', '2024-02-05', '2024-03-18', '2024-05-01', '2024-09-16',
    '2024-11-18', '2024-12-25',
    '2025-01-01', '2025-02-03', '2025-03-17', '2025-05-01', '2025-09-16',
    '2025-11-17', '2025-12-25',
    '2026-01-01', '2026-02-02', '2026-03-16', '2026-05-01',
    # --- Add any company-specific shutdowns below ---
    # '2024-12-26',  # example: day after Christmas shutdown
]

holidays_df = pd.DataFrame({
    'holiday': 'non_operating_day',
    'ds': pd.to_datetime(holiday_dates),
    'lower_window': 0,   # effect starts on the day itself
    'upper_window': 0    # effect ends on the day itself
})

print(f"Holiday entries defined: {len(holidays_df)}")
holidays_df.head()

Holiday entries defined: 25


,holiday,ds,lower_window,upper_window
0,non_operating_day,2023-01-01,0,0
1,non_operating_day,2023-02-06,0,0
2,non_operating_day,2023-03-20,0,0
3,non_operating_day,2023-05-01,0,0
4,non_operating_day,2023-09-16,0,0


# fit model

In [7]:
# ── Initialise model ───────────────────────────────────────────────────────────
model = Prophet(
    # --- Trend ---
    growth='linear',               # use 'logistic' if volume has a known ceiling
    changepoint_prior_scale=0.05,  # flexibility of trend changes; increase if trend
                                   # is under-fit, decrease if over-fit (default 0.05)
    n_changepoints=25,             # number of potential trend-change points

    # --- Seasonality ---
    seasonality_mode='multiplicative',  # good when amplitude scales with trend level
                                        # use 'additive' if amplitude stays constant
    weekly_seasonality=True,
    yearly_seasonality=True,
    daily_seasonality=False,       # daily data → no sub-daily seasonality

    # --- Holidays ---
    holidays=holidays_df,

    # --- Uncertainty ---
    interval_width=0.90,           # 90% confidence interval for truck planning buffer
)

# ── Fit ────────────────────────────────────────────────────────────────────────
model.fit(prophet_df)
print('Model fitted ✅')

11:57:32 - cmdstanpy - INFO - Chain [1] start processing
11:57:33 - cmdstanpy - INFO - Chain [1] done processing


Model fitted ✅


## forecast of 60 days

In [8]:
FORECAST_DAYS = 60

future = model.make_future_dataframe(periods=FORECAST_DAYS, freq='D')
print(f"Future dataframe: {len(future)} rows ({len(prophet_df)} history + {FORECAST_DAYS} forecast)")
future.tail()

Future dataframe: 1141 rows (1081 history + 60 forecast)


,ds
1136,2026-04-25
1137,2026-04-26
1138,2026-04-27
1139,2026-04-28
1140,2026-04-29


## make prediction

In [9]:
# ── Generate predictions ───────────────────────────────────────────────────────
forecast = model.predict(future)

# Clip negative predictions to 0 (volume cannot be negative)
forecast[['yhat', 'yhat_lower', 'yhat_upper']] = (
    forecast[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
)

In [10]:
# Isolate the 60-day forward window
cutoff_date = prophet_df['ds'].max()
forecast_future = forecast[forecast['ds'] > cutoff_date].copy()

print(f"Forecast window: {forecast_future['ds'].min().date()} → {forecast_future['ds'].max().date()}")
forecast_future[['ds','yhat_lower','yhat','yhat_upper']].head(10).round(1)

Forecast window: 2026-03-01 → 2026-04-29


,ds,yhat_lower,yhat,yhat_upper
1081,2026-03-01,0.0,0.0,77.2
1082,2026-03-02,0.0,71.3,157.9
1083,2026-03-03,0.0,88.4,174.3
1084,2026-03-04,0.8,97.7,181.4
1085,2026-03-05,9.0,97.1,185.4
1086,2026-03-06,17.7,103.6,191.1
1087,2026-03-07,0.0,57.1,146.3
1088,2026-03-08,0.0,0.0,85.8
1089,2026-03-09,0.0,72.6,156.3
1090,2026-03-10,1.0,89.6,176.2


In [11]:
plot_plotly(model, forecast)

In [12]:
plot_components_plotly(model, forecast)

# weekly averages

In [25]:
weekly_summary = forecast_future.groupby(forecast_future['ds'].dt.isocalendar().week).agg({
    'yhat': 'mean',
    'yhat_lower': 'mean',
    'yhat_upper': 'mean'
}).reset_index()

current_week = pd.Timestamp(datetime.now()).isocalendar().week

weekly_summary['week'] = weekly_summary['week'] - current_week + 1

truck_daily_capacity = 15  # adjust based on actual truck capacity

weekly_summary['trucks_needed'] = np.ceil(weekly_summary['yhat'] / truck_daily_capacity)
weekly_summary['trucks_needed_lower'] = np.ceil(weekly_summary['yhat_lower'] / truck_daily_capacity)
weekly_summary['trucks_needed_upper'] = np.ceil(weekly_summary['yhat_upper'] / truck_daily_capacity)
weekly_summary.head()

,week,yhat,yhat_lower,yhat_upper,trucks_needed,trucks_needed_lower,trucks_needed_upper
0,0,0.000000,0.000000,77.189610,0.0,0.0,6.0
1,1,73.591941,3.936881,160.311499,5.0,1.0,11.0
2,2,74.326314,5.161639,161.108238,5.0,1.0,11.0
3,3,61.983902,4.317760,148.120293,5.0,1.0,10.0
4,4,65.874402,0.610124,155.616928,5.0,1.0,11.0
